In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Prerequisites ===
# Conda environment yolo-train with packages, registered as a Jupyter kernel.
"""
conda create -n yolo-train python=3.10 -y
conda activate yolo-train

conda install -y -c conda-forge \
  ipykernel numpy pandas pyyaml tqdm matplotlib scikit-learn psutil

pip install --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 \
  torch torchvision torchaudio

pip install --no-cache-dir \
  ultralytics opencv-python-headless seaborn tensorboard ipywidgets \
  "faster-coco-eval>=1.6.7" statsmodels scipy

python -m ipykernel install --user --name yolo-train --display-name "yolo-train"
"""

In [ ]:
# === Cell 3: Imports ===

%matplotlib inline

import os
import sys
import json
import math
import random
import platform
import re
import time
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import yaml

import matplotlib.pyplot as plt
import seaborn as sns

import cv2
import torch
import ultralytics
from ultralytics import YOLO

import ipywidgets as widgets
from IPython.display import display, clear_output

from tqdm.auto import tqdm

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"]      = 120
plt.rcParams["savefig.dpi"]     = 150
plt.rcParams["figure.figsize"]  = (8, 5)
plt.rcParams["axes.titlesize"]  = 12
plt.rcParams["axes.labelsize"]  = 11
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

In [ ]:
# === Cell 4: Config ===

# --- Project paths ---
EXPORT_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/01_LS_Export/20260518_Export")
TRAIN_ROOT  = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation/20260528_Yolo_Hyperparameter_Tuning_yolov8n")

IMAGES_DIR = EXPORT_ROOT / "images" / "all"
LABELS_DIR = EXPORT_ROOT / "labels" / "all"

OBJECT_SPLIT_CSV = EXPORT_ROOT / "object_split.csv"
METADATA_AGG_CSV = EXPORT_ROOT / "metadata_aggregated.csv"
BASE_DATA_YAML   = EXPORT_ROOT / "data.yaml"

# --- Output folders ---
TRAIN_ROOT.mkdir(parents=True, exist_ok=True)

SPLITS_DIR    = TRAIN_ROOT / "splits"
RUNS_DIR      = TRAIN_ROOT / "runs"
RESULTS_DIR   = TRAIN_ROOT / "results"
INFERENCE_DIR = TRAIN_ROOT / "inference"

for p in [SPLITS_DIR, RUNS_DIR, RESULTS_DIR, INFERENCE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PROJECT_NAME    = TRAIN_ROOT.name
DATE_STR        = PROJECT_NAME[:8]
EXPERIMENT_NAME = PROJECT_NAME[9:] if len(PROJECT_NAME) > 9 and PROJECT_NAME[8] == "_" else PROJECT_NAME[8:]
EXPERIMENT_LABEL = "20260528_Yolo_Hyperparameter_Tuning"

# --- Dataset semantics ---
CLASS_MAP = {
    "body":  0,
    "cap":   1,
    "label": 2,
}
MATERIAL_CLASSES = ["HDPE", "PET", "PP", "PS"]

# --- Reproducibility and execution ---
RANDOM_SEED = 42
# Single GPU index; pin screening and HPO to different GPUs to run them in parallel.
DEVICE      = 0
WORKERS     = 4

# --- Cross-validation configuration ---
NUM_FOLDS    = 4
FOLDS_TO_RUN = "all"      # "all" or list of fold indices

# --- YOLO model selection (best model from screening) ---
SELECTED_MODEL = "yolov8n-seg.pt"
TASK           = "segment"

# --- Two-stage HPO setup ---
# Stage A: the Ultralytics tuner proposes TUNE_ITERATIONS candidates on one fold.
# Stage B: each candidate is retrained for TUNE_EPOCHS epochs on all NUM_FOLDS folds.
TUNE_ITERATIONS         = 50
TUNE_EPOCHS             = 50
TOP_N_TRIALS_FOR_PLOTS  = 5
USE_DEFAULT_ULTRALYTICS_TUNER = True

# --- Core tuning and training settings ---
# Fixed batch size; AutoBatch selects batch ~136 for ~240-320 training images,
# which leaves only 2 gradient steps per epoch.
BATCH_SIZE = 16
IMAGE_SIZE = 640
IMAGE_RECT = True

# --- Metrics for reporting and model selection ---
BEST_EPOCH_METRIC           = "metrics/mAP50-95(M)"
BEST_MODEL_SELECTION_METRIC = "metrics/mAP50-95(M)"
F1_METRIC_NAME              = "metrics/F1(M)"

# --- Inference thresholds (used by the visualization widget) ---
CONF_THRESHOLD = 0.25
IOU_THRESHOLD  = 0.50
MAX_DET        = 300

# --- Output filenames ---
BEST_HYPERPARAMETERS_RAW_CSV   = RESULTS_DIR / f"{DATE_STR}_{EXPERIMENT_NAME}_best_hyperparameters_raw.csv"
BEST_TRIAL_HYPERPARAMETERS_CSV = RESULTS_DIR / f"{DATE_STR}_{EXPERIMENT_NAME}_best_trial_hyperparameters.csv"
TUNING_BARPLOT_PNG             = RESULTS_DIR / f"{DATE_STR}_{EXPERIMENT_NAME}_tuning_barplot.png"
TUNING_SCATTERPLOT_PNG         = RESULTS_DIR / f"{DATE_STR}_{EXPERIMENT_NAME}_tuning_scatterplot.png"
METADATA_JSON                  = RESULTS_DIR / f"{DATE_STR}_{EXPERIMENT_NAME}_metadata.json"

# --- Deterministic seeds ---
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# --- Print configuration ---
print("Configuration")
print("-" * 80)
print("EXPORT_ROOT:                  ", EXPORT_ROOT)
print("TRAIN_ROOT:                   ", TRAIN_ROOT)
print("IMAGES_DIR:                   ", IMAGES_DIR)
print("LABELS_DIR:                   ", LABELS_DIR)
print("OBJECT_SPLIT_CSV:             ", OBJECT_SPLIT_CSV)
print("METADATA_AGG_CSV:             ", METADATA_AGG_CSV)
print("BASE_DATA_YAML:               ", BASE_DATA_YAML)
print("SPLITS_DIR:                   ", SPLITS_DIR)
print("RUNS_DIR:                     ", RUNS_DIR)
print("RESULTS_DIR:                  ", RESULTS_DIR)
print("INFERENCE_DIR:                ", INFERENCE_DIR)
print("PROJECT_NAME:                 ", PROJECT_NAME)
print("DATE_STR:                     ", DATE_STR)
print("EXPERIMENT_NAME:              ", EXPERIMENT_NAME)
print("EXPERIMENT_LABEL:             ", EXPERIMENT_LABEL)
print("CLASS_MAP:                    ", CLASS_MAP)
print("MATERIAL_CLASSES:             ", MATERIAL_CLASSES)
print("RANDOM_SEED:                  ", RANDOM_SEED)
print("DEVICE:                       ", DEVICE)
print("WORKERS:                      ", WORKERS)
print("NUM_FOLDS:                    ", NUM_FOLDS)
print("FOLDS_TO_RUN:                 ", FOLDS_TO_RUN)
print("SELECTED_MODEL:               ", SELECTED_MODEL)
print("TASK:                         ", TASK)
print("TUNE_ITERATIONS:              ", TUNE_ITERATIONS)
print("TUNE_EPOCHS:                  ", TUNE_EPOCHS)
print("TOP_N_TRIALS_FOR_PLOTS:       ", TOP_N_TRIALS_FOR_PLOTS)
print("USE_DEFAULT_ULTRALYTICS_TUNER:", USE_DEFAULT_ULTRALYTICS_TUNER)
print("BATCH_SIZE:                   ", BATCH_SIZE)
print("IMAGE_SIZE:                   ", IMAGE_SIZE)
print("IMAGE_RECT:                   ", IMAGE_RECT)
print("BEST_EPOCH_METRIC:            ", BEST_EPOCH_METRIC)
print("BEST_MODEL_SELECTION_METRIC:  ", BEST_MODEL_SELECTION_METRIC)
print("F1_METRIC_NAME:               ", F1_METRIC_NAME)
print("CONF_THRESHOLD:               ", CONF_THRESHOLD)
print("IOU_THRESHOLD:                ", IOU_THRESHOLD)
print("MAX_DET:                      ", MAX_DET)

In [ ]:
# === Cell 5: Sanity Check ===

print("Python:            ", sys.version)
print("Platform:          ", platform.platform())
print("torch version:     ", torch.__version__)
print("CUDA available:    ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count: ", torch.cuda.device_count())
    for d in range(torch.cuda.device_count()):
        print(f"CUDA device {d}:     {torch.cuda.get_device_name(d)}")
print("ultralytics version:", ultralytics.__version__)

PATHS_TO_CHECK = [
    ("EXPORT_ROOT",      EXPORT_ROOT),
    ("TRAIN_ROOT",       TRAIN_ROOT),
    ("IMAGES_DIR",       IMAGES_DIR),
    ("LABELS_DIR",       LABELS_DIR),
    ("OBJECT_SPLIT_CSV", OBJECT_SPLIT_CSV),
    ("METADATA_AGG_CSV", METADATA_AGG_CSV),
    ("BASE_DATA_YAML",   BASE_DATA_YAML),
    ("SPLITS_DIR",       SPLITS_DIR),
    ("RUNS_DIR",         RUNS_DIR),
    ("RESULTS_DIR",      RESULTS_DIR),
    ("INFERENCE_DIR",    INFERENCE_DIR),
]

all_paths_ok = True
print("\nPath checks")
for path_name, path_value in PATHS_TO_CHECK:
    exists = path_value.exists()
    kind   = "DIR " if path_value.is_dir() else "FILE"
    status = "OK     " if exists else "MISSING"
    print(f"  [{kind}] {status}  {path_name}: {path_value}")
    if path_name in {"EXPORT_ROOT", "IMAGES_DIR", "LABELS_DIR",
                     "OBJECT_SPLIT_CSV", "BASE_DATA_YAML"} and not exists:
        all_paths_ok = False

# --- Load object_split.csv and print dataset stats ---
objectsplitdf = None
imagepaths    = list(IMAGES_DIR.glob("*")) if IMAGES_DIR.exists() else []

if OBJECT_SPLIT_CSV.exists():
    objectsplitdf = pd.read_csv(OBJECT_SPLIT_CSV, dtype={"object_id": str})
    objectsplitdf.columns = [c.strip() for c in objectsplitdf.columns]

    print("\nobject_split.csv columns:", list(objectsplitdf.columns))

    uniqueobjects = objectsplitdf.drop_duplicates(subset=["material", "object_id"])
    print("\nDataset stats")
    print("  Distinct objects:  ", len(uniqueobjects))
    print("  Distinct materials:", uniqueobjects["material"].nunique())
    print("  Total images:      ", len(imagepaths))
    display(uniqueobjects.groupby("material")["object_id"].nunique().rename("n_objects"))
else:
    raise FileNotFoundError(f"object_split.csv not found: {OBJECT_SPLIT_CSV}")

# --- Model smoke test ---
try:
    YOLO(SELECTED_MODEL)
    print(f"\nModel instantiation OK: {SELECTED_MODEL}")
except Exception as exc:
    raise RuntimeError(f"Model instantiation failed for {SELECTED_MODEL}") from exc

# --- GPU check for DEVICE ---
if isinstance(DEVICE, int):
    if not torch.cuda.is_available():
        raise RuntimeError(f"DEVICE={DEVICE} requested but no CUDA device is available.")
    if DEVICE >= torch.cuda.device_count():
        raise RuntimeError(
            f"DEVICE={DEVICE} requested but only {torch.cuda.device_count()} GPU(s) visible."
        )
    print(f"\nPinned to GPU {DEVICE}: {torch.cuda.get_device_name(DEVICE)}")

if not all_paths_ok:
    raise RuntimeError("One or more required paths are missing. Fix config before continuing.")

print("\nSanity check PASS")

In [ ]:
# === Cell 6: Filename Parsing ===

# Expected filename pattern: HDPE_obj_01_per_4_2026-04-21.png

FILENAME_REGEX = re.compile(
    r"^(?P<material>[A-Za-z0-9]+)"
    r"_obj_(?P<objectid>\d+)"
    r"_per_(?P<perspectiveid>\d+)"
    r"_(?P<date>\d{4}-\d{2}-\d{2})"
    r"\.(?P<extension>png|jpg|jpeg)$",
    re.IGNORECASE,
)

def parse_material_object_perspective(filename: str):
    """Parse 'HDPE_obj_01_per_4_2026-04-21.png' into (material, objectid, perspectiveid, datestr)."""
    match = FILENAME_REGEX.match(filename)
    if not match:
        raise ValueError(f"Filename does not match expected pattern: {filename}")
    return (match.group("material"),
            match.group("objectid"),
            match.group("perspectiveid"),
            match.group("date"))

object_to_images = defaultdict(list)
bad_filenames    = []

for image_path in imagepaths:
    filename = image_path.name
    try:
        material, objectid, perspectiveid, datestr = parse_material_object_perspective(filename)
        object_to_images[(material, objectid)].append(image_path)
    except ValueError:
        bad_filenames.append(filename)

print("Mapped objects from filenames:", len(object_to_images))

if bad_filenames:
    print(f"\nWARNING: {len(bad_filenames)} filenames did not match expected pattern:")
    for bad_name in bad_filenames[:10]:
        print("  -", bad_name)
    if len(bad_filenames) > 10:
        print(f"  ... and {len(bad_filenames) - 10} more")

if objectsplitdf is not None:
    uniqueobjects  = objectsplitdf.drop_duplicates(subset=["material", "object_id"])
    split_objects  = set(zip(uniqueobjects["material"], uniqueobjects["object_id"]))
    mapped_objects = set(object_to_images.keys())

    missing_in_images = split_objects - mapped_objects
    extra_in_images   = mapped_objects - split_objects

    print("\nConsistency checks")
    print("  Objects in object_split.csv but missing in images:", len(missing_in_images))
    print("  Objects in images but missing in object_split.csv:", len(extra_in_images))

    if missing_in_images:
        print("  Example missing:", list(missing_in_images)[:10])
    if extra_in_images:
        print("  Example extra:  ", list(extra_in_images)[:10])

    if len(missing_in_images) > 0:
        print("\nWARNING: some objects in the split CSV have no images — check IMAGES_DIR path.")
    else:
        print("\nFilename mapping OK — all split objects have images.")

In [ ]:
# === Cell 7: CV Fold Builder ===

random.seed(RANDOM_SEED)

cvdf      = objectsplitdf[objectsplitdf["split"] == "cv"].copy()
testdf    = objectsplitdf[objectsplitdf["split"] == "test"].copy()
skippeddf = objectsplitdf[objectsplitdf["split"] == "skipped"].copy()

def objects_to_image_paths(object_keys):
    """Convert iterable of (material, object_id) tuples to a sorted list of image paths."""
    paths = []
    for material, object_id in object_keys:
        images = object_to_images.get((material, object_id), [])
        if not images:
            print(f"WARNING: no images found for object {(material, object_id)}")
        paths.extend(images)
    return sorted(paths)

def build_split_summary_df(df, split_name):
    """Return a per-material summary dataframe with TOTAL row appended."""
    rows = []
    if df.empty:
        return pd.DataFrame(columns=["split", "material", "objects", "images"])
    for material in sorted(df["material"].unique()):
        material_df   = df[df["material"] == material]
        material_keys = list(zip(material_df["material"], material_df["object_id"]))
        rows.append({
            "split":    split_name,
            "material": material,
            "objects":  material_df["object_id"].nunique(),
            "images":   len(objects_to_image_paths(material_keys)),
        })
    summary_df = pd.DataFrame(rows)
    total_row  = pd.DataFrame([{
        "split":    split_name,
        "material": "TOTAL",
        "objects":  summary_df["objects"].sum(),
        "images":   summary_df["images"].sum(),
    }])
    return pd.concat([summary_df, total_row], ignore_index=True)

print("Split summary by material")
split_summary_df = pd.concat([
    build_split_summary_df(cvdf,      "CV"),
    build_split_summary_df(testdf,    "Test"),
    build_split_summary_df(skippeddf, "Skipped"),
], ignore_index=True)
display(split_summary_df)

# --- Fixed test set (also used by the visualization widget) ---
test_object_keys = list(zip(testdf["material"], testdf["object_id"]))
test_images      = objects_to_image_paths(test_object_keys)

TESTTXT = SPLITS_DIR / "test.txt"
with TESTTXT.open("w", encoding="utf-8") as f:
    for image_path in test_images:
        f.write(str(image_path.resolve()) + "\n")

# --- Object-level stratified K-fold per material ---
def make_folds_for_material(objectids, num_folds=NUM_FOLDS):
    """Shuffle objects and split as evenly as possible across num_folds."""
    objectids = list(objectids)
    random.shuffle(objectids)

    num_total = len(objectids)
    if num_total == 0:
        return [{"train": [], "val": []} for _ in range(num_folds)]
    if num_total < 2:
        print(f"WARNING: Material has only {num_total} CV object; no meaningful train/val split possible.")

    fold_sizes = [num_total // num_folds] * num_folds
    for i in range(num_total % num_folds):
        fold_sizes[i] += 1

    partitions, start = [], 0
    for fold_size in fold_sizes:
        end = start + fold_size
        partitions.append(objectids[start:end])
        start = end

    fold_defs = []
    for fold_index in range(num_folds):
        val_ids   = partitions[fold_index]
        train_ids = [obj for i, part in enumerate(partitions) if i != fold_index for obj in part]
        fold_defs.append({"train": train_ids, "val": val_ids})
    return fold_defs

materials          = sorted(cvdf["material"].unique())
folds              = defaultdict(dict)
material_fold_rows = []

for material in materials:
    material_cv_df     = cvdf[cvdf["material"] == material]
    material_objectids = sorted(material_cv_df["object_id"].unique())
    material_folds     = make_folds_for_material(material_objectids, num_folds=NUM_FOLDS)

    for fold_index, fold_def in enumerate(material_folds):
        train_keys_local = [(material, oid) for oid in fold_def["train"]]
        val_keys_local   = [(material, oid) for oid in fold_def["val"]]

        material_fold_rows.append({
            "fold":         f"fold_{fold_index + 1}",
            "material":     material,
            "trainobjects": len(fold_def["train"]),
            "trainimages":  len(objects_to_image_paths(train_keys_local)),
            "valobjects":   len(fold_def["val"]),
            "valimages":    len(objects_to_image_paths(val_keys_local)),
        })

        folds[fold_index][material] = fold_def

print("Per-material fold summary")
display(pd.DataFrame(material_fold_rows))

with open(BASE_DATA_YAML, "r", encoding="utf-8") as f:
    base_data = yaml.safe_load(f)

FOLD_CONFIGS      = {}
fold_summary_rows = []

for fold_index in range(NUM_FOLDS):
    fold_name = f"fold_{fold_index + 1}"
    fold_dir  = SPLITS_DIR / fold_name
    fold_dir.mkdir(parents=True, exist_ok=True)

    train_keys, val_keys = [], []
    for material in materials:
        train_keys.extend((material, oid) for oid in folds[fold_index][material]["train"])
        val_keys.extend((material, oid)   for oid in folds[fold_index][material]["val"])

    train_images = objects_to_image_paths(train_keys)
    val_images   = objects_to_image_paths(val_keys)

    train_txt = fold_dir / "train.txt"
    val_txt   = fold_dir / "val.txt"

    with train_txt.open("w", encoding="utf-8") as f:
        for image_path in train_images:
            f.write(str(image_path.resolve()) + "\n")

    with val_txt.open("w", encoding="utf-8") as f:
        for image_path in val_images:
            f.write(str(image_path.resolve()) + "\n")

    fold_data          = dict(base_data)
    fold_data["path"]  = str(EXPORT_ROOT.resolve())
    fold_data["train"] = str(train_txt.resolve())
    fold_data["val"]   = str(val_txt.resolve())
    fold_data["test"]  = str(TESTTXT.resolve())

    fold_yaml = SPLITS_DIR / f"data_{fold_name}.yaml"
    with fold_yaml.open("w", encoding="utf-8") as f:
        yaml.safe_dump(fold_data, f, sort_keys=False)

    FOLD_CONFIGS[fold_index] = {
        "name":             fold_name,
        "train_txt":        train_txt,
        "val_txt":          val_txt,
        "test_txt":         TESTTXT,
        "data_yaml":        fold_yaml,
        "num_train_images": len(train_images),
        "num_val_images":   len(val_images),
        "num_test_images":  len(test_images),
        "num_train_objects": len(train_keys),
        "num_val_objects":  len(val_keys),
    }

    fold_summary_rows.append({
        "fold":         fold_name,
        "trainobjects": len(train_keys),
        "trainimages":  len(train_images),
        "valobjects":   len(val_keys),
        "valimages":    len(val_images),
        "testobjects":  len(test_object_keys),
        "testimages":   len(test_images),
        "datayaml":     str(fold_yaml),
    })

print("Fold summary")
display(pd.DataFrame(fold_summary_rows))

print("Finished writing splits and fold data.yaml files.")

In [ ]:
# === Cell 8: Tuning Helpers ===

# --- Safe attribute access (attributes may be methods or plain arrays) ---
def _safe_get(obj, *attrs, default=np.nan):
    """Walk obj.attr1.attr2... safely; call any callable encountered."""
    try:
        cur = obj
        for a in attrs:
            cur = getattr(cur, a)
            if callable(cur) and not isinstance(cur, np.ndarray):
                cur = cur()
        return cur
    except Exception:
        return default

def _safe_float(obj, *attrs, default=np.nan):
    """_safe_get coerced to float."""
    val = _safe_get(obj, *attrs, default=default)
    try:
        return float(val)
    except Exception:
        return default

def _find_col(df, patterns):
    """First column whose name contains any pattern (case-insensitive)."""
    for p in patterns:
        for c in df.columns:
            if p.lower() in c.lower():
                return c
    return None

# --- Model name parsing ---
def parse_model_version_size(model_name: str):
    """Extract (model_version, model_size) from filenames like 'yolo26n-seg.pt'."""
    stem = Path(model_name).stem
    base = stem.split("-")[0]
    if not base.startswith("yolo") or len(base) < 6:
        return base, None
    return base[:-1], base[-1]

# --- results.csv reading (tolerates metric column-name variants) ---
REPORT_METRIC_CANDIDATES = {
    "precision": ["metrics/precision(M)", "metrics/precision(B)", "metrics/precision"],
    "recall":    ["metrics/recall(M)",    "metrics/recall(B)",    "metrics/recall"],
    "map50":     ["metrics/mAP50(M)",     "metrics/mAP50(B)",     "metrics/mAP50"],
    "map50_95":  ["metrics/mAP50-95(M)", "metrics/mAP50-95(B)", "metrics/mAP50-95"],
    "primary":   [BEST_MODEL_SELECTION_METRIC, BEST_EPOCH_METRIC],
}

def resolve_metric_column(columns, candidates):
    """Return the first column from `candidates` that is present in `columns`."""
    column_set = set(columns)
    for candidate in candidates:
        if candidate in column_set:
            return candidate
    return None

def read_best_epoch_row(results_csv_path: Path) -> pd.Series:
    """Read results.csv and return the row of the best epoch by primary metric."""
    if not results_csv_path.exists():
        raise FileNotFoundError(f"results.csv not found: {results_csv_path}")

    df = pd.read_csv(results_csv_path)
    df.columns = [str(c).strip() for c in df.columns]

    metric_col = resolve_metric_column(df.columns, REPORT_METRIC_CANDIDATES["primary"])
    if metric_col is None:
        raise RuntimeError(
            f"Could not find primary metric column in {results_csv_path}. "
            f"Available columns: {list(df.columns)}"
        )
    best_idx = pd.to_numeric(df[metric_col], errors="coerce").idxmax()
    return df.loc[best_idx]

def extract_metric_set(best_row: pd.Series) -> dict:
    """Pull primary + standard auxiliary metrics from one results.csv row."""
    row_columns  = list(best_row.index)
    primary_col  = resolve_metric_column(row_columns, REPORT_METRIC_CANDIDATES["primary"])
    precision_col= resolve_metric_column(row_columns, REPORT_METRIC_CANDIDATES["precision"])
    recall_col   = resolve_metric_column(row_columns, REPORT_METRIC_CANDIDATES["recall"])
    map50_col    = resolve_metric_column(row_columns, REPORT_METRIC_CANDIDATES["map50"])
    map50_95_col = resolve_metric_column(row_columns, REPORT_METRIC_CANDIDATES["map50_95"])

    return {
        "primary_metric_name":  primary_col,
        "primary_metric_value": float(best_row[primary_col])  if primary_col  else np.nan,
        "precision":            float(best_row[precision_col]) if precision_col else np.nan,
        "recall":               float(best_row[recall_col])    if recall_col    else np.nan,
        "map50":                float(best_row[map50_col])     if map50_col     else np.nan,
        "map50_95":             float(best_row[map50_95_col])  if map50_95_col  else np.nan,
    }

# --- Tuner output reader (handles both .ndjson and .csv formats) ---
def read_tune_candidates(tune_dir: Path) -> pd.DataFrame:
    """Load Stage A candidate hyperparameter vectors from a tune run directory."""
    ndjson_path = tune_dir / "tune_results.ndjson"
    rows = []

    if ndjson_path.exists():
        with ndjson_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                hp       = row.get("hyperparameters", {})
                trial_id = row.get("iteration", row.get("trial_id", None))
                fitness  = row.get("fitness", np.nan)
                rec = {"trial_id":      int(trial_id),
                       "tuner_fitness": float(fitness) if fitness is not None else np.nan}
                rec.update(hp)
                rows.append(rec)
        df = pd.DataFrame(rows).sort_values("trial_id").reset_index(drop=True)
        return df

    tune_csv = tune_dir / "tune_results.csv"
    if not tune_csv.exists():
        alt = tune_dir / "tuneresults.csv"
        if alt.exists():
            tune_csv = alt
    if not tune_csv.exists():
        raise FileNotFoundError(f"No tune_results.ndjson or tune_results.csv in {tune_dir}")

    df = pd.read_csv(tune_csv)
    df.columns = [str(c).strip() for c in df.columns]
    if "trial_id" not in df.columns:
        df["trial_id"] = df["iteration"].astype(int) if "iteration" in df.columns \
                         else np.arange(1, len(df) + 1, dtype=int)
    if "fitness" in df.columns and "tuner_fitness" not in df.columns:
        df["tuner_fitness"] = df["fitness"]
    return df.sort_values("trial_id").reset_index(drop=True)

# --- Hyperparameter type normalization ---
# Ultralytics expects some keys as int/bool; tuner output may store them as float/str.
def normalize_hyperparameter_types(hparams: dict) -> dict:
    """Cast known Ultralytics cfg keys back to expected Python types before train()."""
    int_keys = {
        "close_mosaic", "mask_ratio", "workers", "seed", "nbs",
        "epochs", "patience", "save_period", "batch", "imgsz",
    }
    bool_keys = {
        "rect", "val", "plots", "save", "cos_lr", "overlap_mask",
        "single_cls", "deterministic", "amp", "augment", "agnostic_nms",
        "half", "cache",
    }
    normalized = {}
    for key, value in hparams.items():
        if pd.isna(value):
            continue
        value = value.item() if hasattr(value, "item") else value
        if key in int_keys:
            normalized[key] = int(round(value))
        elif key in bool_keys:
            if isinstance(value, str):
                normalized[key] = value.strip().lower() in {"1", "true", "yes", "y"}
            else:
                normalized[key] = bool(value)
        else:
            normalized[key] = value
    return normalized

# --- Tuner-output columns that are not hyperparameters ---
EXCLUDED_CANDIDATE_COLUMNS = {
    "trial_id", "tuner_fitness", "fitness", "iteration",
    "generation", "train_time", "timestamp",
}

print("Tuning helpers loaded.")

In [ ]:
# === Cell 9: Stage A Proposal Tuning ===
# Optimizer: AdamW in all stages. tune() has no optimizer argument and uses the
# automatic selection (AdamW); Stage B and final training set optimizer="AdamW",
# so tuned lr0/momentum values are scored under the same optimizer.

import json
import shutil
from pathlib import Path

# --- Seeds ---
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# --- Folds, model and run names ---
if FOLDS_TO_RUN == "all":
    folds_to_run = sorted(FOLD_CONFIGS.keys())
else:
    folds_to_run = sorted(int(f) for f in FOLDS_TO_RUN)

if len(folds_to_run) == 0:
    raise ValueError("No folds selected. Check FOLDS_TO_RUN.")

model_name                = SELECTED_MODEL
model_base                = Path(model_name).stem
model_version, model_size = parse_model_version_size(model_name)

print("Detected GPUs:", torch.cuda.device_count())
print("Using DEVICE:  ", DEVICE)
print("Using WORKERS: ", WORKERS)
print("Model:         ", model_name, f"(version={model_version}, size={model_size})")

# The tuner runs on one proposal fold only; Stage B evaluates candidates on all folds.
proposal_fold_index = folds_to_run[0]
proposal_fold_name  = FOLD_CONFIGS[proposal_fold_index]["name"]
proposal_data_yaml  = FOLD_CONFIGS[proposal_fold_index]["data_yaml"]
proposal_tune_name  = f"{EXPERIMENT_NAME}_{model_base}_{proposal_fold_name}_proposal_tune"
proposal_tune_dir_base = RUNS_DIR / proposal_tune_name

print("\nStage A — generate candidate hyperparameter vectors")
print("  proposal fold:  ", proposal_fold_name)
print("  tune run name:  ", proposal_tune_name)
print("  tune dir:       ", proposal_tune_dir_base)
print("  TUNE_ITERATIONS:", TUNE_ITERATIONS)
print("  TUNE_EPOCHS:    ", TUNE_EPOCHS)


# --- Tune output directory lookup ---
# YOLO auto-increments the run folder (proposal_tune2, ...) if the base dir exists;
# return the first candidate that contains tune_results.csv or .ndjson.
def _resolve_tune_output_dir(base_dir: Path, max_suffix: int = 20) -> Path:
    candidates = [base_dir] + [
        base_dir.parent / f"{base_dir.name}{i}" for i in range(2, max_suffix + 1)
    ]
    for candidate in candidates:
        if not candidate.is_dir():
            continue
        if (candidate / "tune_results.csv").exists():
            return candidate
        if (candidate / "tune_results.ndjson").exists():
            return candidate
    return base_dir  # no results found; the reader below raises


# --- Remove an empty run directory left by an aborted run ---
# Otherwise YOLO auto-increments the run name.
if proposal_tune_dir_base.exists():
    _has_results = (
        (proposal_tune_dir_base / "tune_results.csv").exists()
        or (proposal_tune_dir_base / "tune_results.ndjson").exists()
    )
    if not _has_results:
        shutil.rmtree(proposal_tune_dir_base)
        print(f"\n  Removed empty stub dir: {proposal_tune_dir_base}")
        print("  (YOLO left this from a prior aborted run; removing prevents name drift)")


# --- Launch tuner ---
proposal_model = YOLO(model_name)
proposal_model.tune(
    data=str(proposal_data_yaml),
    epochs=TUNE_EPOCHS,
    iterations=TUNE_ITERATIONS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    project=str(RUNS_DIR),
    name=proposal_tune_name,
    exist_ok=True,
    val=True,
    rect=IMAGE_RECT,
    seed=RANDOM_SEED,
    deterministic=True,
    plots=False,
    save=False,
    # space=tune_space,   # optional: restrict the search space here
)


# --- Resolve the output directory YOLO wrote to ---
proposal_tune_dir = _resolve_tune_output_dir(proposal_tune_dir_base)

if proposal_tune_dir != proposal_tune_dir_base:
    print(f"\n  WARNING: YOLO wrote results to auto-incremented dir:")
    print(f"    expected : {proposal_tune_dir_base}")
    print(f"    actual   : {proposal_tune_dir}")
    print(f"  This is harmless — results loaded from the actual dir.")
else:
    print(f"\n  Tune dir resolved correctly: {proposal_tune_dir}")

# Use the resolved directory name as the proposal tune name in Stage B.
proposal_tune_name = proposal_tune_dir.name


# --- Parse candidate vectors ---
candidate_trials_df = read_tune_candidates(proposal_tune_dir)

if candidate_trials_df.empty:
    raise RuntimeError("No candidate trials were read from the proposal tune run.")

candidate_hyperparameter_columns = [
    c for c in candidate_trials_df.columns
    if c not in EXCLUDED_CANDIDATE_COLUMNS
]

print("\nStage A complete")
print("  candidate trials found: ", len(candidate_trials_df))
print("  hyperparameter columns:", candidate_hyperparameter_columns)
display(candidate_trials_df.head())

In [ ]:
# === Cell 10: Stage B CV Evaluation ===

# Retrains every Stage A candidate on all folds. Uses candidate_trials_df from the
# Stage A cell, or reads a completed Stage A run from disk.

# --- Recover candidate_trials_df from disk if needed ---
if "candidate_trials_df" not in globals() or candidate_trials_df is None or candidate_trials_df.empty:
    # Stage A run to reuse; edit to point at a different run.
    proposal_tune_name_for_reuse = f"{EXPERIMENT_NAME}_{Path(SELECTED_MODEL).stem}_fold_1_proposal_tune"
    proposal_tune_dir_for_reuse  = RUNS_DIR / proposal_tune_name_for_reuse
    if not proposal_tune_dir_for_reuse.exists():
        raise FileNotFoundError(
            f"Stage A proposal tune directory not found: {proposal_tune_dir_for_reuse}\n"
            "Run Cell 9 first, or edit proposal_tune_name_for_reuse to point at an existing run."
        )
    print(f"Reusing Stage A proposal tune run: {proposal_tune_dir_for_reuse}")
    candidate_trials_df = read_tune_candidates(proposal_tune_dir_for_reuse)
    candidate_hyperparameter_columns = [
        c for c in candidate_trials_df.columns if c not in EXCLUDED_CANDIDATE_COLUMNS
    ]
    proposal_tune_name = proposal_tune_name_for_reuse

if "folds_to_run" not in globals():
    if FOLDS_TO_RUN == "all":
        folds_to_run = sorted(FOLD_CONFIGS.keys())
    else:
        folds_to_run = sorted(int(f) for f in FOLDS_TO_RUN)

if len(folds_to_run) != NUM_FOLDS:
    raise ValueError(
        f"Expected {NUM_FOLDS} folds for full CV, got {len(folds_to_run)}: {folds_to_run}"
    )

model_name = SELECTED_MODEL
model_base = Path(model_name).stem

print("Stage B — CV evaluation of", len(candidate_trials_df), "candidate trials")
print("  model:         ", model_name)
print("  num folds:     ", NUM_FOLDS)
print("  DEVICE:        ", DEVICE)
print("  WORKERS:       ", WORKERS)
print("  TUNE_EPOCHS:   ", TUNE_EPOCHS)
print("  optimizer:     AdamW  (matches Stage A and Final Training; see Cell 9)")
print()

# --- Main Stage B loop ---
cv_trial_rows  = []
per_class_rows = []

for candidate_index, candidate_row in candidate_trials_df.iterrows():
    trial_id      = int(candidate_row["trial_id"])
    timestamp_now = datetime.now().isoformat()

    # Collect non-NaN hyperparameters and normalize types for Ultralytics
    raw_kwargs = {}
    for column in candidate_hyperparameter_columns:
        value = candidate_row[column]
        if pd.isna(value):
            continue
        raw_kwargs[column] = value.item() if hasattr(value, "item") else value
    hyperparameter_kwargs = normalize_hyperparameter_types(raw_kwargs)

    trial_row = {
        "trial_id":           trial_id,
        "timestamp":          timestamp_now,
        "proposal_tune_name": proposal_tune_name,
        "tuner_fitness":      float(candidate_row["tuner_fitness"])
                              if "tuner_fitness" in candidate_row and pd.notna(candidate_row["tuner_fitness"])
                              else np.nan,
        "device":             DEVICE,
        "workers":            WORKERS,
        "optimizer":          "AdamW",
    }
    trial_row.update(hyperparameter_kwargs)

    print(f"\n[Trial {trial_id}] ({candidate_index + 1}/{len(candidate_trials_df)})")

    for fold_index in folds_to_run:
        fold_config = FOLD_CONFIGS[fold_index]
        fold_name   = fold_config["name"]
        fold_number = fold_index + 1

        train_run_name = f"{EXPERIMENT_NAME}_{model_base}_trial_{trial_id:03d}_{fold_name}"
        train_run_dir  = RUNS_DIR / train_run_name

        print(f"  fold {fold_number}/{NUM_FOLDS}: {fold_name}")

        fold_model = YOLO(model_name)
        # AdamW, as in Stage A and final training (see the Stage A cell).
        fold_model.train(
            data=str(fold_config["data_yaml"]),
            epochs=TUNE_EPOCHS,
            patience=TUNE_EPOCHS,        # no early stopping; full epoch budget
            batch=BATCH_SIZE,            # fixed batch size (see Config)
            imgsz=IMAGE_SIZE,
            device=DEVICE,
            workers=WORKERS,
            project=str(RUNS_DIR),       # keeps all runs under RUNS_DIR
            name=train_run_name,
            exist_ok=True,
            task=TASK,
            rect=IMAGE_RECT,
            seed=RANDOM_SEED,
            deterministic=True,
            val=True,
            plots=False,
            save=False,
            verbose=False,
            optimizer="AdamW",           # same optimizer in all stages
            **hyperparameter_kwargs,
        )

        # --- Best-epoch row and standard metric set ---
        results_csv_path = train_run_dir / "results.csv"
        best_row    = read_best_epoch_row(results_csv_path)
        metric_set  = extract_metric_set(best_row)

        trial_row[f"fold_{fold_number}_metric"]    = metric_set["primary_metric_value"]
        trial_row[f"fold_{fold_number}_precision"] = metric_set["precision"]
        trial_row[f"fold_{fold_number}_recall"]    = metric_set["recall"]
        trial_row[f"fold_{fold_number}_map50"]     = metric_set["map50"]
        trial_row[f"fold_{fold_number}_map50_95"]  = metric_set["map50_95"]

        # --- Per-class metrics ---
        # seg.all_ap is a plain ndarray [n_classes, n_iou_thresholds]; column 0 is AP50.
        try:
            best_weights = train_run_dir / "weights" / "best.pt"
            if best_weights.exists():
                eval_model   = YOLO(str(best_weights))
                eval_results = eval_model.val(
                    data=str(fold_config["data_yaml"]),
                    split="val",
                    imgsz=IMAGE_SIZE,
                    conf=CONF_THRESHOLD,
                    iou=IOU_THRESHOLD,
                    device=DEVICE,
                    workers=WORKERS,
                    rect=IMAGE_RECT,
                    plots=False,
                    verbose=False,
                    project=str(RUNS_DIR),
                    name=f"{train_run_name}__val_perclass",
                    exist_ok=True,
                )
                seg          = _safe_get(eval_results, "seg", default=None)
                names_map    = eval_results.names
                ap_class_idx = np.asarray(_safe_get(seg, "ap_class_index"))
                all_ap       = np.asarray(_safe_get(seg, "all_ap"))
                if all_ap.ndim == 2 and len(ap_class_idx) > 0:
                    ap50_pc   = all_ap[:, 0]
                    ap5095_pc = all_ap.mean(axis=1)
                    p_pc = np.asarray(_safe_get(seg, "p", default=np.full(len(ap_class_idx), np.nan)))
                    r_pc = np.asarray(_safe_get(seg, "r", default=np.full(len(ap_class_idx), np.nan)))
                    for k, cls_idx in enumerate(ap_class_idx):
                        cls_name = names_map.get(int(cls_idx), str(cls_idx))
                        p_k = float(p_pc[k]) if k < len(p_pc) else np.nan
                        r_k = float(r_pc[k]) if k < len(r_pc) else np.nan
                        f1_k = (2 * p_k * r_k / (p_k + r_k)) if (not np.isnan(p_k + r_k) and (p_k + r_k) > 0) else np.nan
                        per_class_rows.append({
                            "trial_id":  trial_id,
                            "fold_idx":  fold_index,
                            "fold_name": fold_name,
                            "class_id":  int(cls_idx),
                            "class":     cls_name,
                            "AP50":      round(float(ap50_pc[k]),   6),
                            "AP50-95":   round(float(ap5095_pc[k]), 6),
                            "precision": round(p_k,  6),
                            "recall":    round(r_k,  6),
                            "F1":        round(f1_k, 6) if not np.isnan(f1_k) else np.nan,
                        })
        except Exception as e:
            print(f"    per-class extraction skipped: {e}")

    cv_trial_rows.append(trial_row.copy())

if not cv_trial_rows:
    raise RuntimeError("No CV trial rows were collected.")

cv_trials_df = pd.DataFrame(cv_trial_rows)

# Save per-class Stage B metrics
if per_class_rows:
    per_class_csv = RESULTS_DIR / f"{DATE_STR}_{EXPERIMENT_NAME}_per_class_stageB.csv"
    pd.DataFrame(per_class_rows).to_csv(per_class_csv, index=False)
    print(f"\nSaved per-class Stage B metrics to: {per_class_csv}")

print("\nStage B complete:", len(cv_trials_df), "trials evaluated across", NUM_FOLDS, "folds.")
display(cv_trials_df.head())

In [ ]:
# === Cell 11: Tuning Plots and Best-Trial Export ===

# --- Local helpers ---
def _pick_existing(columns, candidates):
    """Return the first column in `candidates` that exists in `columns`."""
    column_set = set(columns)
    for candidate in candidates:
        if candidate in column_set:
            return candidate
    return None

def _safe_numeric(value):
    """Coerce a single value to numeric, returning NaN on failure."""
    return pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]

def _parse_trial_fold_from_run_name(run_name: str):
    """Extract (trial_id, fold_number) from an Ultralytics run-name string."""
    patterns = [
        re.compile(r"trial_?0*(\d+)_fold_?0*(\d+)", re.IGNORECASE),
        re.compile(r"trial_?0*(\d+).*?_fold_?0*(\d+)", re.IGNORECASE),
    ]
    for pattern in patterns:
        match = pattern.search(run_name)
        if match:
            return int(match.group(1)), int(match.group(2))
    return None, None

def _build_fold_wide(df_long: pd.DataFrame, value_column: str, suffix: str) -> pd.DataFrame:
    """Pivot a long-form fold dataframe into wide format with `fold_K_<suffix>` columns."""
    wide = (df_long.pivot(index="trial_id", columns="fold_number", values=value_column)
                   .sort_index(axis=1).reset_index())
    wide.columns = ["trial_id" if c == "trial_id" else f"fold_{int(c)}_{suffix}"
                    for c in wide.columns]
    return wide

def _validate_complete_fold_columns(df: pd.DataFrame, suffix: str, num_folds: int):
    """Ensure all expected `fold_K_<suffix>` columns are present and non-null."""
    expected = [f"fold_{k}_{suffix}" for k in range(1, num_folds + 1)]
    missing  = [c for c in expected if c not in df.columns]
    if missing:
        raise RuntimeError(f"Missing expected {suffix} columns: {missing}")
    missing_values_mask = df[expected].isna()
    if missing_values_mask.any().any():
        bad_ids = df.loc[missing_values_mask.any(axis=1), "trial_id"].tolist()
        raise RuntimeError(f"Missing values in {suffix} columns. Affected trial_ids: {bad_ids}")
    return expected

def _first_non_null(series: pd.Series):
    """Return the first non-null entry of a series, or NaN."""
    non_null = series.dropna()
    return non_null.iloc[0] if len(non_null) else np.nan

# --- Rebuild Stage B results from runs/ on disk (e.g. after a kernel restart) ---
def rebuild_raw_trials_from_runs():
    """Reconstruct raw_trials_df by scanning RUNS_DIR for completed Stage B runs."""
    runs_dir = RUNS_DIR

    primary_candidates = [BEST_MODEL_SELECTION_METRIC, BEST_EPOCH_METRIC,
                           "metrics/mAP50-95(M)", "metrics/mAP50(M)", "fitness"]
    aux_candidates = {
        "precision": ["metrics/precision(M)", "metrics/precision(B)", "metrics/precision"],
        "recall":    ["metrics/recall(M)",    "metrics/recall(B)",    "metrics/recall"],
        "map50":     ["metrics/mAP50(M)",     "metrics/mAP50(B)",     "metrics/mAP50"],
        "map50_95":  ["metrics/mAP50-95(M)", "metrics/mAP50-95(B)", "metrics/mAP50-95"],
    }
    known_hp_keys = ["lr0", "lrf", "momentum", "weight_decay", "warmup_epochs",
                     "warmup_momentum", "box", "cls", "dfl"]
    excluded_arg_keys = {"model", "data", "project", "name", "save_dir", "task", "mode",
                          "exist_ok", "plots", "save", "verbose", "val"}
    generic_config_keys = {"epochs", "batch", "imgsz", "rect", "device", "workers", "seed",
                            "augment", "optimizer", "patience", "cache", "amp", "cos_lr",
                            "single_cls", "pretrained", "deterministic", "close_mosaic",
                            "overlap_mask", "mask_ratio", "dropout", "resume", "time", "freeze"}
    rows = []
    for results_csv in sorted(Path(runs_dir).rglob("results.csv")):
        run_dir   = results_csv.parent
        args_yaml = run_dir / "args.yaml"

        trial_id, fold_number = _parse_trial_fold_from_run_name(run_dir.name)
        if trial_id is None or fold_number is None:
            trial_id, fold_number = _parse_trial_fold_from_run_name(run_dir.as_posix())
        if trial_id is None or fold_number is None:
            continue
        if not args_yaml.exists():
            continue
        try:
            with open(args_yaml, "r", encoding="utf-8") as f:
                args = yaml.safe_load(f) or {}
        except Exception:
            continue
        try:
            df_results = pd.read_csv(results_csv)
        except Exception:
            continue
        if df_results.empty:
            continue
        df_results.columns = [str(c).strip() for c in df_results.columns]

        primary_col = _pick_existing(df_results.columns, primary_candidates)
        if primary_col is None:
            continue
        prim_series = pd.to_numeric(df_results[primary_col], errors="coerce")
        if prim_series.notna().sum() == 0:
            continue
        best_idx = prim_series.idxmax()
        best_row = df_results.loc[best_idx]

        row = {
            "trial_id":       int(trial_id),
            "fold_number":    int(fold_number),
            "timestamp":      datetime.fromtimestamp(results_csv.stat().st_mtime).isoformat(),
            "metric_name":    primary_col,
            "primary_metric": _safe_numeric(best_row.get(primary_col, np.nan)),
        }
        for alias, cands in aux_candidates.items():
            col = _pick_existing(df_results.columns, cands)
            row[alias] = _safe_numeric(best_row.get(col, np.nan)) if col else np.nan

        for k, v in args.items():
            if k in excluded_arg_keys:
                continue
            if isinstance(v, (str, int, float, bool)) or v is None:
                row[k] = v
        rows.append(row)

    if not rows:
        raise RuntimeError(f"No completed Stage B results.csv files found under {runs_dir}")

    stage_b_long = (pd.DataFrame(rows)
                      .sort_values(["trial_id", "fold_number", "timestamp"])
                      .reset_index(drop=True))
    dup = stage_b_long.duplicated(subset=["trial_id", "fold_number"], keep="last")
    if dup.any():
        stage_b_long = stage_b_long.loc[~dup].copy()

    expected_folds = set(range(1, NUM_FOLDS + 1))
    coverage = (stage_b_long.groupby("trial_id")["fold_number"]
                              .apply(lambda s: sorted(set(int(x) for x in s.dropna().tolist())))
                              .to_dict())
    incomplete = {int(t): sorted(expected_folds - set(fs))
                  for t, fs in coverage.items() if set(fs) != expected_folds}
    if incomplete:
        raise RuntimeError(f"Incomplete CV coverage detected: {incomplete}")

    present_known_hp = [c for c in known_hp_keys if c in stage_b_long.columns]
    extras = []
    for col in stage_b_long.columns:
        if col in {"trial_id", "fold_number", "timestamp", "metric_name",
                   "primary_metric", "precision", "recall", "map50", "map50_95"}:
            continue
        if col in present_known_hp or col in generic_config_keys:
            continue
        if not pd.api.types.is_numeric_dtype(stage_b_long[col]):
            continue
        per_trial = stage_b_long.groupby("trial_id")[col].nunique(dropna=True)
        if len(per_trial) == 0 or per_trial.max() > 1:
            continue
        if stage_b_long[col].nunique(dropna=True) > 1:
            extras.append(col)
    hyperparameter_columns_local = present_known_hp + sorted(extras)

    trial_base_agg = {c: _first_non_null for c in hyperparameter_columns_local}
    trial_base_agg["timestamp"] = "max"
    trial_base = stage_b_long.groupby("trial_id", as_index=False).agg(trial_base_agg).copy()

    metric_wide    = _build_fold_wide(stage_b_long, "primary_metric", "metric")
    precision_wide = _build_fold_wide(stage_b_long, "precision",      "precision")
    recall_wide    = _build_fold_wide(stage_b_long, "recall",         "recall")
    map50_wide     = _build_fold_wide(stage_b_long, "map50",          "map50")
    map50_95_wide  = _build_fold_wide(stage_b_long, "map50_95",       "map50_95")

    raw = (trial_base
           .merge(metric_wide,    on="trial_id", how="inner")
           .merge(precision_wide, on="trial_id", how="left")
           .merge(recall_wide,    on="trial_id", how="left")
           .merge(map50_wide,     on="trial_id", how="left")
           .merge(map50_95_wide,  on="trial_id", how="left")
           .copy())

    expected_metric_cols   = _validate_complete_fold_columns(raw, "metric",   NUM_FOLDS)
    expected_map5095_cols  = _validate_complete_fold_columns(raw, "map50_95", NUM_FOLDS)

    raw["metric_mean"]   = raw[expected_metric_cols].mean(axis=1)
    raw["metric_std"]    = raw[expected_metric_cols].std(axis=1, ddof=1)
    raw["map50_95_mean"] = raw[expected_map5095_cols].mean(axis=1)
    raw["map50_95_std"]  = raw[expected_map5095_cols].std(axis=1, ddof=1)

    raw = (raw.sort_values(["metric_mean", "metric_std", "trial_id"],
                           ascending=[False, True, True])
              .reset_index(drop=True).copy())

    resolved = sorted(stage_b_long["metric_name"].dropna().unique().tolist())
    primary_metric_name = (BEST_MODEL_SELECTION_METRIC
                            if BEST_MODEL_SELECTION_METRIC in resolved
                            else (resolved[0] if resolved else BEST_MODEL_SELECTION_METRIC))
    return raw, primary_metric_name, hyperparameter_columns_local

# --- Resolve raw_trials_df (in-memory > saved CSV > rebuild from disk) ---
PROTECTED_AGG_COLS = {
    "trial_id", "timestamp", "metric_name", "primary_metric",
    "metric_mean", "metric_std",
    "precision_mean", "precision_std", "recall_mean", "recall_std",
    "map50_mean", "map50_std", "map50_95_mean", "map50_95_std",
    "tuner_fitness", "proposal_tune_name", "device", "workers", "optimizer",
}

def _build_raw_from_cv_trials(cv_df: pd.DataFrame) -> pd.DataFrame:
    """Add fold mean/std columns to the Stage B results and sort by metric."""
    raw = cv_df.copy()
    metric_cols = [f"fold_{k}_metric"   for k in range(1, NUM_FOLDS + 1) if f"fold_{k}_metric"   in raw.columns]
    map_cols    = [f"fold_{k}_map50_95" for k in range(1, NUM_FOLDS + 1) if f"fold_{k}_map50_95" in raw.columns]
    raw["metric_mean"]   = raw[metric_cols].mean(axis=1) if metric_cols else np.nan
    raw["metric_std"]    = raw[metric_cols].std(axis=1, ddof=1) if metric_cols else np.nan
    raw["map50_95_mean"] = raw[map_cols].mean(axis=1) if map_cols else np.nan
    raw["map50_95_std"]  = raw[map_cols].std(axis=1, ddof=1) if map_cols else np.nan
    raw = (raw.sort_values(["metric_mean", "metric_std", "trial_id"],
                           ascending=[False, True, True])
              .reset_index(drop=True))
    return raw

if "cv_trials_df" in globals() and isinstance(cv_trials_df, pd.DataFrame) and not cv_trials_df.empty:
    raw_trials_df = _build_raw_from_cv_trials(cv_trials_df)
    PRIMARY_METRIC_NAME    = BEST_MODEL_SELECTION_METRIC
    hyperparameter_columns = [c for c in raw_trials_df.columns
                              if c not in PROTECTED_AGG_COLS
                              and not str(c).startswith("fold_")]
elif BEST_HYPERPARAMETERS_RAW_CSV.exists():
    print(f"Loading cached raw trials from: {BEST_HYPERPARAMETERS_RAW_CSV}")
    raw_trials_df = pd.read_csv(BEST_HYPERPARAMETERS_RAW_CSV)
    PRIMARY_METRIC_NAME    = BEST_MODEL_SELECTION_METRIC
    hyperparameter_columns = [c for c in raw_trials_df.columns
                              if c not in PROTECTED_AGG_COLS
                              and not str(c).startswith("fold_")]
else:
    print("BEST_HYPERPARAMETERS_RAW_CSV not found. Rebuilding from runs/ ...")
    raw_trials_df, PRIMARY_METRIC_NAME, hyperparameter_columns = rebuild_raw_trials_from_runs()

raw_trials_df.to_csv(BEST_HYPERPARAMETERS_RAW_CSV, index=False)
print(f"Saved raw trials table to: {BEST_HYPERPARAMETERS_RAW_CSV}")
print(f"PRIMARY_METRIC_NAME: {PRIMARY_METRIC_NAME}")

# --- Top-N plots ---
plot_df = raw_trials_df.head(min(TOP_N_TRIALS_FOR_PLOTS, len(raw_trials_df))).copy()
plot_df["trial_label"] = plot_df["trial_id"].astype(int).astype(str)

if "map50_95_mean" not in plot_df.columns or "map50_95_std" not in plot_df.columns:
    raise RuntimeError("map50_95_mean / map50_95_std not available for plotting.")

# --- Plot 1: top-N bar chart (mean ± std) ---
fig_bar, ax_bar = plt.subplots(figsize=(max(8, 1.0 * len(plot_df)), 5))
x_positions = np.arange(len(plot_df))
ax_bar.bar(
    x_positions,
    plot_df["map50_95_mean"].values,
    yerr=plot_df["map50_95_std"].fillna(0).values,
    capsize=4, color="tab:blue", alpha=0.85,
)
ax_bar.set_xticks(x_positions)
ax_bar.set_xticklabels(plot_df["trial_label"], rotation=45, ha="right")
ax_bar.set_xlabel("Trial ID")
ax_bar.set_ylabel("mAP50-95 mean across folds")
ax_bar.set_title(f"Top {len(plot_df)} trials: mAP50-95 mean ± std across folds")
ax_bar.grid(axis="y", alpha=0.3)
for x_pos, value in zip(x_positions, plot_df["map50_95_mean"].values):
    ax_bar.text(x_pos, value, f"{value:.3f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
fig_bar.savefig(TUNING_BARPLOT_PNG, bbox_inches="tight", dpi=150)
plt.show()
print("Saved tuning bar plot to:", TUNING_BARPLOT_PNG)

# --- Plot 2: top-N fold-level scatter + trial mean ---
map_fold_cols = [f"fold_{k}_map50_95" for k in range(1, NUM_FOLDS + 1)
                 if f"fold_{k}_map50_95" in plot_df.columns]
if not map_fold_cols:
    raise RuntimeError("No fold-level mAP50-95 columns found for scatter plot.")

fig_scatter, ax_scatter = plt.subplots(figsize=(max(8, 1.0 * len(plot_df)), 5))
for plot_index, (_, row) in enumerate(plot_df.iterrows()):
    fold_vals = [row[c] for c in map_fold_cols if pd.notna(row[c])]
    if fold_vals:
        jitter = np.random.uniform(-0.06, 0.06, size=len(fold_vals))
        ax_scatter.scatter(
            np.full(len(fold_vals), plot_index) + jitter,
            fold_vals, s=28, alpha=0.45, color="tab:blue",
            label="Fold mAP50-95" if plot_index == 0 else None,
        )
    ax_scatter.scatter(
        plot_index, row["map50_95_mean"], s=90, color="tab:orange",
        edgecolor="black", zorder=3,
        label="Trial mean" if plot_index == 0 else None,
    )
ax_scatter.set_xticks(np.arange(len(plot_df)))
ax_scatter.set_xticklabels(plot_df["trial_label"], rotation=45, ha="right")
ax_scatter.set_xlabel("Trial ID")
ax_scatter.set_ylabel("mAP50-95")
ax_scatter.set_title(f"Top {len(plot_df)} trials: fold mAP50-95 values + mean")
ax_scatter.grid(axis="y", alpha=0.3)
ax_scatter.legend()
plt.tight_layout()
fig_scatter.savefig(TUNING_SCATTERPLOT_PNG, bbox_inches="tight", dpi=150)
plt.show()
print("Saved tuning scatter plot to:", TUNING_SCATTERPLOT_PNG)

# --- Best trial: rank by metric_mean desc, then metric_std asc ---
if raw_trials_df.empty:
    raise RuntimeError("raw_trials_df is empty, cannot extract best trial hyperparameters.")

raw_trials_df_sorted = raw_trials_df.sort_values(
    ["metric_mean", "metric_std", "trial_id"],
    ascending=[False, True, True],
).reset_index(drop=True)
best_trial_row = raw_trials_df_sorted.iloc[0].copy()

candidate_hp_cols = [c for c in raw_trials_df_sorted.columns
                     if c not in PROTECTED_AGG_COLS and not str(c).startswith("fold_")]
candidate_hp_cols = [c for c in candidate_hp_cols
                     if c in best_trial_row.index and not pd.isna(best_trial_row[c])]

if not candidate_hp_cols:
    print("WARNING: No hyperparameter columns found in best_trial_row.")
    best_trial_hyperparameters_df = pd.DataFrame(columns=["trial_id", "hyperparameter", "value"])
else:
    best_trial_hyperparameters_df = pd.DataFrame([
        {"trial_id":       int(best_trial_row["trial_id"]),
         "hyperparameter": column,
         "value":          best_trial_row[column]}
        for column in sorted(candidate_hp_cols)
    ])

best_trial_hyperparameters_df = (
    best_trial_hyperparameters_df
    .sort_values(["trial_id", "hyperparameter"])
    .reset_index(drop=True)
)

# Best-trial hyperparameters; input to the final training notebook
best_trial_hyperparameters_df.to_csv(BEST_TRIAL_HYPERPARAMETERS_CSV, index=False)

print("Best trial ID:", int(best_trial_row["trial_id"]))
print("Saved best-trial hyperparameters CSV to:", BEST_TRIAL_HYPERPARAMETERS_CSV)
print("(Format: trial_id, hyperparameter, value — consumed verbatim by Notebook 4.)")
display(best_trial_hyperparameters_df)

# --- Metadata JSON ---
metadata_payload = {
    "created_at":                  datetime.now().isoformat(),
    "project_name":                PROJECT_NAME,
    "experiment_name":             EXPERIMENT_NAME,
    "experiment_label":            EXPERIMENT_LABEL,
    "selected_model":              SELECTED_MODEL,
    "best_trial_id":               int(best_trial_row["trial_id"]),
    "best_trial_metric_mean":      float(best_trial_row.get("metric_mean", np.nan)),
    "best_trial_metric_std":       float(best_trial_row.get("metric_std",  np.nan)),
    "best_trial_map50_95_mean":    float(best_trial_row.get("map50_95_mean", np.nan)),
    "best_trial_map50_95_std":     float(best_trial_row.get("map50_95_std",  np.nan)),
    "num_folds":                   NUM_FOLDS,
    "tune_iterations":             TUNE_ITERATIONS,
    "tune_epochs":                 TUNE_EPOCHS,
    "batch_size":                  BATCH_SIZE,
    "image_size":                  IMAGE_SIZE,
    "device":                      DEVICE,
    "workers":                     WORKERS,
    "stage_b_optimizer":           "SGD",
    "best_epoch_metric":           BEST_EPOCH_METRIC,
    "best_model_selection_metric": BEST_MODEL_SELECTION_METRIC,
    "best_trial_hyperparameters_csv": str(BEST_TRIAL_HYPERPARAMETERS_CSV),
    "best_hyperparameters_raw_csv":   str(BEST_HYPERPARAMETERS_RAW_CSV),
}
with METADATA_JSON.open("w", encoding="utf-8") as f:
    json.dump(metadata_payload, f, indent=2, ensure_ascii=False)
print("Saved metadata JSON to:", METADATA_JSON)

In [ ]:
# === Cell 12: Best-Trial Visualization Widget ===

# --- Color palettes ---
CLASS_COLORS_RGB = {                  # prediction colors (solid, vivid)
    0: (255,   0,   0),               # body  - red
    1: (  0, 200,   0),               # cap   - green
    2: (  0,  80, 255),               # label - blue
}
GT_COLORS_RGB = {                     # ground-truth colors (lighter pastels)
    0: (255, 160, 160),               # body  - light pink/red
    1: (160, 255, 160),               # cap   - light green
    2: (160, 200, 255),               # label - light blue
}
DEFAULT_COLOR_RGB = (255, 255,   0)
DEFAULT_GT_RGB    = (255, 255, 160)
MASK_ALPHA        = 0.35
BOX_THICKNESS     = 2
TEXT_SCALE        = 0.55
TEXT_THICKNESS    = 2

CLASS_ID_TO_NAME = {v: k for k, v in CLASS_MAP.items()}

def _rgb2bgr(c):   return (c[2], c[1], c[0])
def _cls_bgr(cid): return _rgb2bgr(CLASS_COLORS_RGB.get(int(cid), DEFAULT_COLOR_RGB))
def _gt_bgr(cid):  return _rgb2bgr(GT_COLORS_RGB.get(int(cid), DEFAULT_GT_RGB))

# --- YOLO label parsing for GT overlay ---
def parse_yolo_label_line(line):
    """Parse one YOLO polygon label line into {'class_id', 'polygon_norm'}."""
    parts = line.strip().split()
    if len(parts) < 7:
        return None
    cls_id  = int(float(parts[0]))
    coords  = [float(v) for v in parts[1:]]
    polygon = list(zip(coords[0::2], coords[1::2]))
    return {"class_id": cls_id, "polygon_norm": polygon}

def read_non_empty_lines(path):
    """Return non-empty stripped lines from a text file."""
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

# --- Dashed polygon helper (OpenCV has no native dashed line) ---
def draw_dashed_polygon(img, points, color, thickness=2, dash_px=12, gap_px=7):
    """Draw a closed dashed polygon on img in-place."""
    pts = list(points)
    pts.append(pts[0])
    for i in range(len(pts) - 1):
        x0, y0 = int(pts[i][0]),   int(pts[i][1])
        x1, y1 = int(pts[i+1][0]), int(pts[i+1][1])
        dx, dy  = x1 - x0, y1 - y0
        seg_len = max(1, int(np.hypot(dx, dy)))
        step    = dash_px + gap_px
        t       = 0
        while t < seg_len:
            t_end = min(t + dash_px, seg_len)
            sx = int(round(x0 + dx * t     / seg_len))
            sy = int(round(y0 + dy * t     / seg_len))
            ex = int(round(x0 + dx * t_end / seg_len))
            ey = int(round(y0 + dy * t_end / seg_len))
            cv2.line(img, (sx, sy), (ex, ey), color, thickness, cv2.LINE_AA)
            t += step

# --- Renderer ---
def draw_overlay(image_path, result,
                 show_masks=True, show_contours=True,
                 show_boxes=False, show_labels=True, show_gt=True):
    """Render predictions (solid) + GT (dashed lighter) on one image; return RGB."""
    img_bgr = result.orig_img.copy()
    overlay = img_bgr.copy()
    boxes   = result.boxes
    masks   = result.masks
    h, w    = img_bgr.shape[:2]

    if show_masks and masks is not None and masks.xy is not None:
        for i, poly in enumerate(masks.xy):
            if poly is None or len(poly) < 3:
                continue
            cls_id = int(boxes.cls[i].item()) if boxes is not None and i < len(boxes) else -1
            pts    = np.array(poly, dtype=np.int32).reshape(-1, 1, 2)
            cv2.fillPoly(overlay, [pts], _cls_bgr(cls_id))
    blended = cv2.addWeighted(overlay, MASK_ALPHA, img_bgr, 1 - MASK_ALPHA, 0)

    if boxes is not None and len(boxes):
        xyxy    = boxes.xyxy.cpu().numpy()
        cls_ids = boxes.cls.cpu().numpy().astype(int)
        confs   = boxes.conf.cpu().numpy()
        polys   = masks.xy if masks is not None and masks.xy is not None else []

        for i in range(len(xyxy)):
            cls_id      = int(cls_ids[i])
            conf        = float(confs[i])
            color       = _cls_bgr(cls_id)
            x1, y1, x2, y2 = xyxy[i].astype(int)
            if show_contours and i < len(polys):
                poly = polys[i]
                if poly is not None and len(poly) >= 3:
                    pts = np.array(poly, dtype=np.int32).reshape(-1, 1, 2)
                    cv2.polylines(blended, [pts], True, color, 2, cv2.LINE_AA)
            if show_boxes:
                cv2.rectangle(blended, (x1, y1), (x2, y2), color, BOX_THICKNESS)
            if show_labels:
                name  = CLASS_ID_TO_NAME.get(cls_id, str(cls_id))
                label = f"{name} {conf:.2f}"
                (tw, th), bl = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX,
                                               TEXT_SCALE, TEXT_THICKNESS)
                ly1 = max(0, y1 - th - bl - 4)
                cv2.rectangle(blended, (x1, ly1), (x1 + tw + 6, y1), color, -1)
                cv2.putText(blended, label, (x1 + 3, y1 - 4),
                            cv2.FONT_HERSHEY_SIMPLEX, TEXT_SCALE,
                            (255, 255, 255), TEXT_THICKNESS, cv2.LINE_AA)

    if show_gt:
        label_path = LABELS_DIR / (Path(image_path).stem + ".txt")
        if label_path.exists():
            with label_path.open("r", encoding="utf-8") as f:
                for line in f:
                    parsed = parse_yolo_label_line(line)
                    if parsed is None:
                        continue
                    pts_px = [(int(round(x * w)), int(round(y * h)))
                              for x, y in parsed["polygon_norm"]]
                    if len(pts_px) < 3:
                        continue
                    gt_color = _gt_bgr(parsed["class_id"])
                    draw_dashed_polygon(blended, pts_px, gt_color,
                                        thickness=2, dash_px=12, gap_px=7)

    return cv2.cvtColor(blended, cv2.COLOR_BGR2RGB)

# --- Best HPO trial weights: (trial, fold) with the highest fold metric ---
if "raw_trials_df" not in globals() or raw_trials_df is None or raw_trials_df.empty:
    raise RuntimeError("raw_trials_df not available — run Cell 11 first.")

best_trial_id = int(raw_trials_df.iloc[0]["trial_id"])
model_base    = Path(SELECTED_MODEL).stem

best_fold_index = None
best_fold_value = -np.inf
for k in range(1, NUM_FOLDS + 1):
    col = f"fold_{k}_metric"
    if col in raw_trials_df.columns:
        v = raw_trials_df.iloc[0][col]
        if pd.notna(v) and float(v) > best_fold_value:
            best_fold_value = float(v)
            best_fold_index = k

if best_fold_index is None:
    raise RuntimeError("Could not locate a best fold for the best trial.")

best_run_name = f"{EXPERIMENT_NAME}_{model_base}_trial_{best_trial_id:03d}_fold_{best_fold_index}"
best_run_dir  = RUNS_DIR / best_run_name
best_weights  = best_run_dir / "weights" / "best.pt"

print(f"Best trial: {best_trial_id} | best fold: {best_fold_index}")
print(f"Best run dir: {best_run_dir}")
print(f"Best weights: {best_weights} (exists={best_weights.exists()})")

if not best_weights.exists():
    raise FileNotFoundError(
        f"Best weights not found at {best_weights}. "
        "Stage B may not have completed for this trial+fold."
    )

# --- Image pools: Val (best fold's val.txt) and Test ---
val_pool  = [Path(p) for p in read_non_empty_lines(FOLD_CONFIGS[best_fold_index - 1]["val_txt"])]
test_pool = [Path(p) for p in read_non_empty_lines(TESTTXT)] if TESTTXT.exists() else []

if not val_pool and not test_pool:
    print("No images available for visualization (neither Val nor Test).")
else:
    viz_model = YOLO(str(best_weights))

    mode_toggle = widgets.ToggleButtons(
        options=[("Val images", "val"), ("Test images", "test")],
        value="val" if val_pool else "test",
        description="Mode:", button_style="",
    )

    def _make_opts(pool):
        return [(f"{i}: {p.name}", i) for i, p in enumerate(pool)]

    _init_pool     = val_pool if mode_toggle.value == "val" else test_pool
    image_dropdown = widgets.Dropdown(
        options=_make_opts(_init_pool), description="Image:",
        layout=widgets.Layout(width="700px"),
    )

    show_button   = widgets.Button(description="Show",     button_style="primary", icon="eye")
    random_button = widgets.Button(description="Random",   button_style="info",    icon="random")
    save_button   = widgets.Button(description="Save PNG", icon="save")

    conf_slider = widgets.FloatSlider(
        value=CONF_THRESHOLD, min=0.0, max=1.0, step=0.01,
        description="Conf:", readout_format=".2f",
        layout=widgets.Layout(width="350px"),
    )
    iou_slider = widgets.FloatSlider(
        value=IOU_THRESHOLD, min=0.10, max=0.90, step=0.05,
        description="IoU:", readout_format=".2f",
        layout=widgets.Layout(width="350px"),
    )

    show_masks_cb    = widgets.Checkbox(value=True,  description="Masks")
    show_contours_cb = widgets.Checkbox(value=True,  description="Contours")
    show_boxes_cb    = widgets.Checkbox(value=False, description="Boxes")
    show_labels_cb   = widgets.Checkbox(value=True,  description="Labels")
    show_gt_cb       = widgets.Checkbox(value=True,  description="Ground truth (dashed)")

    viewer_output = widgets.Output()

    def _current_pool():   return val_pool if mode_toggle.value == "val" else test_pool
    def _current_prefix(): return "Val"    if mode_toggle.value == "val" else "Test"

    def _render(save_path=None):
        """Run inference on the selected image and render the overlay."""
        pool = _current_pool()
        idx  = image_dropdown.value
        if idx is None or idx < 0 or idx >= len(pool):
            return
        image_path = pool[idx]
        results = viz_model.predict(
            source=str(image_path), conf=conf_slider.value,
            iou=iou_slider.value, imgsz=IMAGE_SIZE,
            retina_masks=True, verbose=False, save=False, device=DEVICE,
        )
        r       = results[0]
        n_pred  = 0 if r.boxes is None else len(r.boxes)
        rgb     = draw_overlay(
            image_path, r,
            show_masks=show_masks_cb.value, show_contours=show_contours_cb.value,
            show_boxes=show_boxes_cb.value, show_labels=show_labels_cb.value,
            show_gt=show_gt_cb.value,
        )
        with viewer_output:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 8))
            ax.imshow(rgb)
            ax.set_title(
                f"{_current_prefix()}: {image_path.name}  |  pred={n_pred}  |  "
                f"conf>={conf_slider.value:.2f}\n"
                f"trial {best_trial_id} / fold {best_fold_index}  |  "
                f"solid=prediction  dashed=GT  |  "
                f"body=red  cap=green  label=blue  (GT = lighter shade)",
                fontsize=9,
            )
            ax.axis("off")
            plt.tight_layout()
            if save_path:
                fig.savefig(save_path, bbox_inches="tight", dpi=150)
                print(f"Saved: {save_path}")
            plt.show()

    def on_mode_changed(change):
        """Repopulate the dropdown when the Val/Test toggle flips."""
        if change["name"] != "value":
            return
        pool = _current_pool()
        if not pool:
            with viewer_output:
                clear_output(wait=True)
                print(f"No images available for mode '{change['new']}'.")
            image_dropdown.options = []
            image_dropdown.value   = None
        else:
            image_dropdown.options = _make_opts(pool)
            image_dropdown.value   = 0

    mode_toggle.observe(on_mode_changed)
    show_button.on_click(lambda _: _render())
    random_button.on_click(lambda _: (
        setattr(image_dropdown, "value", random.randrange(len(_current_pool()))),
        _render(),
    ))
    save_button.on_click(lambda _: _render(
        save_path=RESULTS_DIR / f"viz_{mode_toggle.value}_trial{best_trial_id:03d}_fold{best_fold_index}"
                                f"_{Path(_current_pool()[image_dropdown.value]).stem}.png"
    ))

    display(
        widgets.VBox([
            widgets.HBox([mode_toggle]),
            widgets.HBox([image_dropdown]),
            widgets.HBox([show_button, random_button, save_button]),
            widgets.HBox([conf_slider, iou_slider]),
            widgets.HBox([show_masks_cb, show_contours_cb, show_boxes_cb,
                          show_labels_cb, show_gt_cb]),
        ]),
        viewer_output,
    )
    on_mode_changed({"name": "value", "new": mode_toggle.value})